# Premier modèles de classification: 
Notre objectif est de contruire et évaluer des premiers modèles permettant de prédire le statut académique  des étudiants : Gradute, Dropout ou Enrolled. 

In [68]:
#Chargement des données:
import pandas as pd

train=pd.read_csv("../data/train.csv")
test=pd.read_csv("../data/test.csv")

print("La dimension du train est: ",train.shape)
print("La dimension du test est: ",test.shape)

La dimension du train est:  (76518, 38)
La dimension du test est:  (51012, 37)


### Separation des variables explicatives et de la variable cible:

In [69]:
x=train.drop(columns=["id","Target"])
y=train["Target"]
print("La dimension de x est: ",x.shape)
print("La dimension de y est: ",y.shape)

La dimension de x est:  (76518, 36)
La dimension de y est:  (76518,)


### Sépartions des données d'entrainements et de validations: 

In [70]:
from sklearn.model_selection import train_test_split
x_train,x_valid,y_train,y_valid=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

print("La dimension du train est: ",x_train.shape)
print("La dimension de validation est: ",x_valid.shape)


La dimension du train est:  (61214, 36)
La dimension de validation est:  (15304, 36)


### Preparation des données pour la régréssion logistique: 

In [71]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
#On calcule la moyenne et l'écart type sur les données du  train puis on les standardise  
x_train_scaled=scaler.fit_transform(x_train)
# on utilise la moyenne et l'écart type calculés sur le train pour standardiser les données de validation
x_valid_scaled=scaler.transform(x_valid)


### La régression logistique :

In [72]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

model=LogisticRegression( max_iter=200,random_state=42)
model.fit(x_train_scaled,y_train)
y_pred=model.predict(x_valid_scaled)

accuracy=accuracy_score(y_valid,y_pred)
print("L'accuracy du modèle avec la régression logistique est: ",round(accuracy, 4))
# Avec l'accuraccy on sait pas exactement dans quelle catégorie le modèle se trompe, on va donc utiliser le classification report pour avoir plus d'informations sur les performances du modèle
print("Classification report: \n",classification_report(y_valid,y_pred))

L'accuracy du modèle avec la régression logistique est:  0.8157
Classification report: 
               precision    recall  f1-score   support

     Dropout       0.89      0.81      0.85      5059
    Enrolled       0.63      0.55      0.59      2988
    Graduate       0.84      0.93      0.88      7257

    accuracy                           0.82     15304
   macro avg       0.78      0.76      0.77     15304
weighted avg       0.81      0.82      0.81     15304



On remarque qu'avec la réfréssion logistique on arrive à un accuracy de 81,57% sur nos données de validation.

Le modèle reconnait assez bien les étudiants qui sont  diplomés(Graduate) et les étudiant qui ont abandonnés(Dropout).
Par contre , il rencontre d'avantage de difficultés  avec les étudiant encore injscrits(Enrolled)  avec une reconnaissance de 55%.
Ainsi ce modèle sera une référence pour le comparer avec les prochains modèles.

### Random Forest : 

In [73]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report
# Création et entraînement du modèle 
model_rf=RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1)
model_rf.fit(x_train,y_train)


pred_rf=model_rf.predict(x_valid)
accuracy_rf=accuracy_score(y_valid,pred_rf)

print("L'accuracy du modèle random forest est: ",round(accuracy_rf, 4))
print("Classification report: \n",classification_report(y_valid,pred_rf))

L'accuracy du modèle random forest est:  0.8283
Classification report: 
               precision    recall  f1-score   support

     Dropout       0.90      0.82      0.86      5059
    Enrolled       0.65      0.60      0.62      2988
    Graduate       0.85      0.93      0.89      7257

    accuracy                           0.83     15304
   macro avg       0.80      0.78      0.79     15304
weighted avg       0.83      0.83      0.83     15304



Ainsi  le modèle Random Forset  obtient une accuracy de 82,83%, contre 81,57% pour le modèle de régression logistique .

On voit qu'il y a  une lègère amélioration notamment pour les étudiant encore inscrit (Enrolled) même si elle reste une catégorie difficile à reconnaitre. 

Donc Random Forest est notre meilleur modèle pour l'instant,avant l'optimisation.

## Modèle XGBoost : 

In [74]:
from sklearn.preprocessing import LabelEncoder

# Encoder la variable cible pour XGBoost
encodeur=LabelEncoder()
y_train_encod=encodeur.fit_transform(y_train)
y_valid_encod=encodeur.transform(y_valid)
print( "Les classes encodées sont: ",encodeur.classes_)


Les classes encodées sont:  ['Dropout' 'Enrolled' 'Graduate']


In [75]:
from xgboost import XGBClassifier

model_xgb=XGBClassifier(n_estimators=200,random_state=42,n_jobs=-1)

model_xgb.fit(x_train,y_train_encod)

y_pred_xgb=model_xgb.predict(x_valid)
accuracy_xgb=accuracy_score(y_valid_encod,y_pred_xgb)
print("L'accuracy du modèle XGBoost est: ",round(accuracy_xgb, 4))
print("Classification report: \n",classification_report(y_valid_encod,y_pred_xgb,target_names=encodeur.classes_))

L'accuracy du modèle XGBoost est:  0.8268
Classification report: 
               precision    recall  f1-score   support

     Dropout       0.89      0.83      0.86      5059
    Enrolled       0.65      0.60      0.62      2988
    Graduate       0.85      0.92      0.89      7257

    accuracy                           0.83     15304
   macro avg       0.80      0.78      0.79     15304
weighted avg       0.82      0.83      0.82     15304



On remarque qu'avec le modèle XGBoost on obtient une accuracy de 82,68% et c'est un résultat qui reste proche de l'accuracy de celui de Random Forest.

Cependant Random Forest reste légérement  meilleur pour l'instant.Nous allons  donc essayer d'optimiser les hyperparamètres  de nos modèles afin  d'améliorer les perfermances.

### Optimisation de la régression logistique : 
#### Comme on a des variables qualitatives et d'autres quantitatives alors on va faire une amélioration du prétraitement des variables:

In [76]:
#Les variables qualitatives nominales
colonnes_cat=["Marital status","Application mode","Course",
"Previous qualification","Nacionality","Mother's qualification",
"Father's qualification","Mother's occupation","Father's occupation"]

#Les variables binaire 
colonnes_bin=["Daytime/evening attendance","Displaced",
"Educational special needs","Debtor","Tuition fees up to date","Gender","Scholarship holder","International"]  

# Autres variables numériques
colonnes_num=[ col for col in x.columns if col not in colonnes_cat and col not in colonnes_bin]

print("Le nombre de variables numériques est: ",len(colonnes_num))
print("Le nombre de variables catégorielles est: ",len(colonnes_cat))
print("Le nombre de variables binaires est: ",len(colonnes_bin))

Le nombre de variables numériques est:  19
Le nombre de variables catégorielles est:  9
Le nombre de variables binaires est:  8


In [78]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler

# preparation des données avec le ColumnTransformer pour appliquer les transformations appropriées à chaque type de variable
Preparation=ColumnTransformer( [ ("numérique",StandardScaler(),colonnes_num), 
("catégorielles",OneHotEncoder(handle_unknown='ignore'),colonnes_cat), ("binaires","passthrough",colonnes_bin) ] )

modele_lr=Pipeline([("preparation",Preparation),("modele",LogisticRegression(max_iter=2000,random_state=42))])
modele_lr.fit(x_train,y_train)
y_pred_lr=modele_lr.predict(x_valid)

accuracy_lr=accuracy_score(y_valid,y_pred_lr)
print("L'accuracy du modèle avec la régression logistique est: ",round(accuracy_lr, 4))
print("Classification report: \n",classification_report(y_valid,y_pred_lr))

L'accuracy du modèle avec la régression logistique est:  0.824
Classification report: 
               precision    recall  f1-score   support

     Dropout       0.89      0.83      0.86      5059
    Enrolled       0.65      0.57      0.61      2988
    Graduate       0.84      0.93      0.88      7257

    accuracy                           0.82     15304
   macro avg       0.79      0.77      0.78     15304
weighted avg       0.82      0.82      0.82     15304



Ainsi en intégrant le prétraitement des variables selon leur types, la regression logistique obtient une accuracy de 82,40% contre 81,57% avec notre première mèthode.

Donc on observe une lègère amélioration des perfermances.Ce qui montre que le traitement des variables catégorielles avec OneHotEncoder  est plus adapté à nos données. 

Cependant le Random Forest reste lègerement meilleur avec une accuracy de 82,83%.

### Optimisation du modèle RandomForest :

In [79]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

#Hyperparamètres à tester pour le Random Forest
parametres={ "n_estimators":[100,200,300],"max_depth":[10,20,None],
"min_samples_split":[2,5,10],"min_samples_leaf":[1,2,4]}

# Recherche des meilleurs hyperparamètres avec RandomizedSearchCV
recherche_rf=RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42,n_jobs=-1), param_distributions=parametres, 
n_iter=10, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)

#faire l'entrainement pour chercher les meilleurs hyperparamètres Du RAndom Forest et en evaluant plusieur modèles . 
recherche_rf.fit(x_train,y_train)
print("Meilleurs hyperparamètres trouvés: ", recherche_rf.best_params_)

print("Accuracy moyenne en validation croisée est: ", round(recherche_rf.best_score_, 4))

meilleur_rf=recherche_rf.best_estimator_
y_pred_rf_opt=meilleur_rf.predict(x_valid)

accuracy_rf_opt=accuracy_score(y_valid,y_pred_rf_opt)
print("L'accuracy du modèle Random Forest optimisé est: ",round(accuracy_rf_opt, 4))

Meilleurs hyperparamètres trouvés:  {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20}
Accuracy moyenne en validation croisée est:  0.8261
L'accuracy du modèle Random Forest optimisé est:  0.8273


Aprés l'optimisation des hyperparamètre avec RandomizedSearchTV, le Random Forest obtient une accuaracy moyenne de 82,61% en validation croisé, et une accuracy de 82,73% d'accuracy sur notre jeu de validation.

On remarque que le résultat obtenue reste légèrement inférieur à celui du Random Forest initial(82,83%) , donc l'optimisation n'a pas permis d'améliorer les perfermonces du modèle pour l'instant. 


### Optimisation du modèle  XGBoost:

In [80]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

#Hyperparamètres à tester
parametres_xgb={ "n_estimators":[100,200,300],"max_depth":[3,5,7],
"learning_rate":[0.01,0.1,0.2]}

# Recherche des meilleurs hyperparamètres avec RandomizedSearchCV
recherche_xgb=RandomizedSearchCV(estimator=XGBClassifier(random_state=42), param_distributions=parametres_xgb, 
n_iter=10, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)

recherche_xgb.fit(x_train,y_train_encod)
print("Meilleurs hyperparamètres trouvés pour XGBoost est : ", recherche_xgb.best_params_)
print("Accuracy moyenne en validation croisée est: ", round(recherche_xgb.best_score_, 4))

meilleur_xgb=recherche_xgb.best_estimator_
y_pred_xgb_opt=meilleur_xgb.predict(x_valid)
accuracy_xgb_opt=accuracy_score(y_valid_encod,y_pred_xgb_opt)
print("L'accuracy du modèle XGBoost optimisé est: ",round(accuracy_xgb_opt, 4))
print("Classification report: \n",classification_report(y_valid_encod,y_pred_xgb_opt,target_names=encodeur.classes_))

Meilleurs hyperparamètres trouvés pour XGBoost est :  {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.2}
Accuracy moyenne en validation croisée est:  0.8308
L'accuracy du modèle XGBoost optimisé est:  0.8292
Classification report: 
               precision    recall  f1-score   support

     Dropout       0.90      0.83      0.86      5059
    Enrolled       0.65      0.60      0.63      2988
    Graduate       0.85      0.92      0.89      7257

    accuracy                           0.83     15304
   macro avg       0.80      0.78      0.79     15304
weighted avg       0.83      0.83      0.83     15304



Aprés l'optimisation des hyperparamètre avec RandomizedSearchTV, le modèle XGBoost obtient une accuaracy moyenne de 83,08% en validation croisé, et une accuracy de 82,92% d'accuracy sur notre jeu de validation.

On observe une petite amélioration par rapport au modèle XGBoost initial (82,68%). Le XGBoost optimisé devient ainsi parmi les plus perforamant de ceux que nous avons tésté , devant le Random Forest initial(82,83%).

Ainsi les écart restent faibles, ce qui ne permet pas encore de conclure que XGBoost est meilleur.

In [81]:
resultats=pd.DataFrame({"Modèle":["Régression logistique","Random Forest","XGBoost",
"Random Forest optimisé","XGBoost optimisé"],
"Accuracy":[accuracy_lr,accuracy_rf,accuracy_xgb,accuracy_rf_opt,accuracy_xgb_opt]})

resultats=resultats.sort_values("Accuracy", ascending=False)
print(resultats)

                   Modèle  Accuracy
4        XGBoost optimisé  0.829195
1           Random Forest  0.828346
3  Random Forest optimisé  0.827300
2                 XGBoost  0.826777
0   Régression logistique  0.824033


#### Conclusion:
Nous avons entrainé et comparer les trois modèles: La regression logistique, le Random Forest et XGBoost.
Les performances obtenues sont assez proches avec des accuracies entre 82,40% et 82,92%.L'optimisation des hyperparamètre n'a pas améliorer l'accuracy de Random Forest , contrairement à XGBoost où on a réaliser une certaine amélioration.

Donc On retient XGBoost optimisé pour l'instant ,puisque il obtient le meilleur résultat  sur notre jeu de validation.
